In [8]:
from rdflib import Graph, Namespace
from pathlib import Path

In [9]:
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

ontology_path = project_folder / "ontology" / "transportation_phase2_validated.owl"

g = Graph()
g.parse(str(ontology_path), format="xml")

print("Ontology loaded successfully!")
print("Total RDF triples:", len(g))

Ontology loaded successfully!
Total RDF triples: 346


In [10]:
TRAN = Namespace("http://example.org/transportation.owl#")

g.bind("tran", TRAN)

prefix = """
PREFIX tran: <http://example.org/transportation.owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
"""

In [11]:
query1 = prefix + """
SELECT ?incident ?road
WHERE {
    ?incident tran:Blocks ?road .
}
"""

for row in g.query(query1):
    print(row.incident.split("#")[-1], "blocks", row.road.split("#")[-1])

Accident_1 blocks Road_C_G


In [12]:
query2 = prefix + """
SELECT ?incident ?congestion
WHERE {
    ?incident tran:Causes ?congestion .
}
"""

for row in g.query(query2):
    print(
        row.incident.split("#")[-1],
        "causes",
        row.congestion.split("#")[-1]
    )

Accident_1 causes Congestion_1


In [13]:
query3 = prefix + """
SELECT ?congestion ?route
WHERE {
    ?congestion tran:Influences ?route .
}
"""

for row in g.query(query3):
    print(
        row.congestion.split("#")[-1],
        "influences",
        row.route.split("#")[-1]
    )

Congestion_1 influences Route_1


In [14]:
query4 = prefix + """
SELECT ?signal ?vehicle
WHERE {
    ?signal tran:Prioritizes ?vehicle .
}
"""

for row in g.query(query4):
    print(
        row.signal.split("#")[-1],
        "prioritizes",
        row.vehicle.split("#")[-1]
    )

Signal_B prioritizes Ambulance_1


In [15]:
query5 = prefix + """
SELECT ?junction
WHERE {
    tran:Road_C_G tran:ConnectedTo ?junction .
}
"""

for row in g.query(query5):
    print(row.junction.split("#")[-1])

Junction_C
Junction_G


In [16]:
query6 = prefix + """
SELECT ?road ?speed
WHERE {
    ?road tran:speedLimit ?speed .
    FILTER(?speed > 50)
}
"""

for row in g.query(query6):
    print(row.road.split("#")[-1], "-", row.speed, "km/h")

Road_A_B - 60 km/h
Road_A_C - 60 km/h
Road_B_D - 60 km/h
Road_B_E - 60 km/h
Road_C_E - 60 km/h
Road_D_F - 60 km/h
Road_F_G - 60 km/h


In [17]:
query7 = prefix + """
SELECT ?vehicle ?location
WHERE {
    ?vehicle tran:LocatedAt ?location .
    FILTER(?vehicle = tran:Ambulance_1)
}
"""

for row in g.query(query7):
    print(
        row.vehicle.split("#")[-1],
        "is located at",
        row.location.split("#")[-1]
    )

Ambulance_1 is located at Junction_A


In [18]:
query8 = prefix + """
SELECT ?incident ?road ?congestion ?route
WHERE {
    ?incident tran:Blocks ?road .
    ?incident tran:Causes ?congestion .
    ?congestion tran:Influences ?route .
}
"""

for row in g.query(query8):

    print("Incident:", row.incident.split("#")[-1])
    print("Blocked Road:", row.road.split("#")[-1])
    print("Congestion:", row.congestion.split("#")[-1])
    print("Affected Route:", row.route.split("#")[-1])

Incident: Accident_1
Blocked Road: Road_C_G
Congestion: Congestion_1
Affected Route: Route_1


In [19]:
queries = [
    query1, query2, query3, query4,
    query5, query6, query7, query8
]

print("SPARQL QUERY VERIFICATION")
print("-------------------------")

all_passed = True

for i, query in enumerate(queries, start=1):
    results = list(g.query(query))

    if results:
        print(f"Query {i}: PASS ({len(results)} results)")
    else:
        print(f"Query {i}: FAIL (No results)")
        all_passed = False

print("\nAll queries returned results:", all_passed)

SPARQL QUERY VERIFICATION
-------------------------
Query 1: PASS (1 results)
Query 2: PASS (1 results)
Query 3: PASS (1 results)
Query 4: PASS (1 results)
Query 5: PASS (2 results)
Query 6: PASS (7 results)
Query 7: PASS (1 results)
Query 8: PASS (1 results)

All queries returned results: True


In [20]:
def names(query):
    return {
        tuple(str(value).split("#")[-1] for value in row)
        for row in g.query(query)
    }

expected = {
    1: {("Accident_1", "Road_C_G")},
    2: {("Accident_1", "Congestion_1")},
    3: {("Congestion_1", "Route_1")},
    4: {("Signal_B", "Ambulance_1")},
    5: {("Junction_C",), ("Junction_G",)},
    7: {("Ambulance_1", "Junction_A")},
    8: {("Accident_1", "Road_C_G", "Congestion_1", "Route_1")}
}

for number, expected_rows in expected.items():
    actual_rows = names(queries[number - 1])
    assert actual_rows == expected_rows, f"Query {number} returned unexpected results"

print("All seven expected-result checks passed!")

All seven expected-result checks passed!


In [21]:
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

export_folder = project_folder / "ontology" / "phase2"

export_folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_folder)
print("Export directory:", export_folder)

Project folder: c:\Users\dellu\Desktop\Transportation-Knowledge-Framework
Export directory: c:\Users\dellu\Desktop\Transportation-Knowledge-Framework\ontology\phase2


In [22]:
owl_file = export_folder / "transportation_v2.owl"

g.serialize(destination=str(owl_file), format="xml")

print("OWL exported successfully!")

OWL exported successfully!


In [23]:
ttl_file = export_folder / "transportation_v2.ttl"

g.serialize(destination=str(ttl_file), format="turtle")

print("Turtle exported successfully!")

Turtle exported successfully!
